# Demucs Source Separation on Google Colab

**Purpose**: Separate Touhou doujin tracks into vocals + instrumentals for style classification.

**Hypothesis**: Liz Triangle ↔ IOSYS confusion is driven by shared vocal characteristics.

**Runtime**: ~2-3 hours for 828 tracks on free T4 GPU

---

## Instructions
1. Upload your audio files to Google Drive (or use the upload cell below)
2. Run cells in order
3. Stems will be saved to Drive for download

**Tip**: Use `Runtime > Change runtime type > T4 GPU` for faster processing

## 1. Setup

In [ ]:
# Install demucs (takes ~2-3 minutes)
!pip install -q demucs

# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Processing will be slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\nDrive mounted at /content/drive/MyDrive/")

## 2. Configure Paths

Edit these paths to match your Drive structure:

In [ ]:
# ============================================================
# EDIT THESE PATHS
# ============================================================

# Where your raw audio files are (in Google Drive)
INPUT_BASE = "/content/drive/MyDrive/touhou_classifier/data/raw"

# Where to save separated stems
OUTPUT_BASE = "/content/drive/MyDrive/touhou_classifier/data/separated"

# Circles to process (folder names)
CIRCLES = [
    "IOSYS",
    "UNDEAD_CORPORATION",
    "Akatsuki_Records",
    "SOUND_HOLIC",
    "Liz_Triangle",
]

# Demucs model (htdemucs is best quality)
MODEL = "htdemucs"

# ============================================================

In [ ]:
# Check what files exist
import os
from pathlib import Path

print("Scanning input directories...\n")

total_files = 0
for circle in CIRCLES:
    circle_path = Path(INPUT_BASE) / circle
    if circle_path.exists():
        files = list(circle_path.glob("*.flac")) + list(circle_path.glob("*.mp3"))
        print(f"{circle}: {len(files)} files")
        total_files += len(files)
    else:
        print(f"{circle}: NOT FOUND at {circle_path}")

print(f"\nTotal: {total_files} files")
print(f"Estimated time: {total_files * 30 // 60} - {total_files * 45 // 60} minutes on T4 GPU")

## 3. Alternative: Upload Files Directly

If you don't want to use Drive, upload files directly (slower for large datasets):

In [ ]:
# OPTIONAL: Upload files directly instead of using Drive
# Uncomment to use:

# from google.colab import files
# uploaded = files.upload()  # Select files from your computer
# 
# # Files will be in /content/
# !ls -la /content/*.flac /content/*.mp3 2>/dev/null || echo "No audio files uploaded"

## 4. Run Demucs Separation

In [ ]:
import subprocess
import shutil
from pathlib import Path
from tqdm import tqdm
import time

def separate_file(input_path, output_dir, model="htdemucs"):
    """
    Separate a single file and combine non-vocal stems into instrumental.
    
    Returns: (vocals_path, instrumental_path) or None on error
    """
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    stem_name = input_path.stem
    vocals_path = output_dir / f"{stem_name}_vocals.wav"
    instrumental_path = output_dir / f"{stem_name}_instrumental.wav"
    
    # Skip if already processed
    if vocals_path.exists() and instrumental_path.exists():
        return "skipped"
    
    # Run demucs
    temp_dir = Path("/content/temp_demucs")
    temp_dir.mkdir(exist_ok=True)
    
    cmd = [
        "python", "-m", "demucs",
        "-n", model,
        "-o", str(temp_dir),
        str(input_path)
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        return f"error: {result.stderr[:100]}"
    
    # Move vocals
    demucs_output = temp_dir / model / input_path.stem
    vocals_src = demucs_output / "vocals.wav"
    
    if vocals_src.exists():
        shutil.copy(vocals_src, vocals_path)
    
    # Combine drums + bass + other into instrumental
    drums_src = demucs_output / "drums.wav"
    bass_src = demucs_output / "bass.wav"
    other_src = demucs_output / "other.wav"
    
    if drums_src.exists() and bass_src.exists() and other_src.exists():
        mix_cmd = [
            "ffmpeg", "-y",
            "-i", str(drums_src),
            "-i", str(bass_src),
            "-i", str(other_src),
            "-filter_complex", "amix=inputs=3:duration=longest",
            str(instrumental_path)
        ]
        subprocess.run(mix_cmd, capture_output=True)
    
    # Cleanup temp
    if demucs_output.exists():
        shutil.rmtree(demucs_output)
    
    return "success"


def process_circle(circle_name, input_base, output_base, model="htdemucs"):
    """Process all files in a circle directory."""
    
    input_dir = Path(input_base) / circle_name
    output_dir = Path(output_base) / circle_name
    
    if not input_dir.exists():
        return {"error": f"Directory not found: {input_dir}"}
    
    files = list(input_dir.glob("*.flac")) + list(input_dir.glob("*.mp3"))
    
    results = {"success": 0, "skipped": 0, "errors": []}
    
    for f in tqdm(files, desc=circle_name):
        status = separate_file(f, output_dir, model)
        
        if status == "success":
            results["success"] += 1
        elif status == "skipped":
            results["skipped"] += 1
        else:
            results["errors"].append((f.name, status))
    
    return results

print("Functions defined. Run the next cell to start processing.")

In [ ]:
# ============================================================
# MAIN PROCESSING LOOP
# This will take 2-3 hours for ~800 tracks
# ============================================================

print("="*60)
print("DEMUCS SOURCE SEPARATION")
print("="*60)
print(f"Model: {MODEL}")
print(f"Input: {INPUT_BASE}")
print(f"Output: {OUTPUT_BASE}")
print("="*60)

start_time = time.time()
total_stats = {"success": 0, "skipped": 0, "errors": 0}

for circle in CIRCLES:
    print(f"\nProcessing {circle}...")
    
    results = process_circle(circle, INPUT_BASE, OUTPUT_BASE, MODEL)
    
    if "error" in results:
        print(f"  ERROR: {results['error']}")
        continue
    
    print(f"  Success: {results['success']}")
    print(f"  Skipped: {results['skipped']}")
    print(f"  Errors: {len(results['errors'])}")
    
    total_stats["success"] += results["success"]
    total_stats["skipped"] += results["skipped"]
    total_stats["errors"] += len(results["errors"])

elapsed = time.time() - start_time

print("\n" + "="*60)
print("COMPLETE")
print("="*60)
print(f"Total processed: {total_stats['success']}")
print(f"Total skipped: {total_stats['skipped']}")
print(f"Total errors: {total_stats['errors']}")
print(f"Time elapsed: {elapsed/60:.1f} minutes")
print(f"\nOutput saved to: {OUTPUT_BASE}")

## 5. Verify Output

In [ ]:
# Check what was created
print("Output summary:\n")

for circle in CIRCLES:
    output_dir = Path(OUTPUT_BASE) / circle
    if output_dir.exists():
        vocals = list(output_dir.glob("*_vocals.wav"))
        instrumental = list(output_dir.glob("*_instrumental.wav"))
        print(f"{circle}:")
        print(f"  Vocals: {len(vocals)} files")
        print(f"  Instrumental: {len(instrumental)} files")
    else:
        print(f"{circle}: No output directory")

In [ ]:
# Listen to a sample (optional)
import IPython.display as ipd

# Pick a sample to preview
sample_circle = "Liz_Triangle"  # Change as needed
sample_dir = Path(OUTPUT_BASE) / sample_circle

if sample_dir.exists():
    vocals = list(sample_dir.glob("*_vocals.wav"))
    if vocals:
        print(f"Sample: {vocals[0].name}")
        print("\nVocals:")
        display(ipd.Audio(str(vocals[0])))
        
        instrumental = sample_dir / vocals[0].name.replace("_vocals", "_instrumental")
        if instrumental.exists():
            print("\nInstrumental:")
            display(ipd.Audio(str(instrumental)))
else:
    print(f"No output for {sample_circle} yet")

## 6. Download Results

If you used Google Drive, files are already synced. Otherwise:

In [ ]:
# OPTIONAL: Zip and download (if not using Drive)

# !zip -r /content/separated_stems.zip {OUTPUT_BASE}
# 
# from google.colab import files
# files.download('/content/separated_stems.zip')

---

## Next Steps

After downloading the separated stems:

1. Copy to your local project:
   ```bash
   cp -r ~/Downloads/separated/* data/separated/
   ```

2. Train on separated sources:
   ```bash
   python scripts/train_separated.py --mode instrumental
   python scripts/train_separated.py --mode vocals
   python scripts/train_separated.py --mode combined
   ```

3. Compare with HPSS baseline to see if neural separation helps more.